# AIMLCZG546 — Software Engineering for Machine Learning
## Assignment II — Implementation, Code Quality and QA

**Group No: 173**

| Sl. No | BITS ID | Name | Contribution (Qualitative) | % |
|---|---|---|---|---|
| 1 | 2025AA05957 | Abhishek | Modular OOP / functional design | 100 |
| 2 | 2025AA05729 | Ashmit Bhandari | Two or more test types/Formatting and linting | 100 |
| 3 | 2025AA05478 | Rishabh Jain | Error handling and logging | 100 |
| 4 | 2025AB05319 | Udit Sharma | REST API / Research vs production code | 100 |

**Project:** Health Insurance Cross-Sell Prediction System
**Repository:** https://github.com/ashmitB-droid/seml2_group173
**API (Assignment II):** https://seml2-group173.onrender.com/docs

---

### How to run this notebook

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements-dev.txt
python -m src.train          # writes models/ and metrics.json
jupyter notebook 173.ipynb
```

Run from the repository root. On macOS, `brew install libomp` is needed for XGBoost.

---
## Setup

In [1]:
import subprocess, sys, json, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def run(cmd):
    """Run a shell command and print its combined output."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    return r.returncode

print("Python", sys.version.split()[0])
print("Working directory:", ROOT)

Python 3.12.3
Working directory: /home/claude/repo2


---
# Objective 1 — Implementation and Code Sharing

## 1. Modular design using OOP and functional principles

The pipeline is split so each module has a single responsibility. `ModelTrainer`
and `Predictor` are classes because both hold state that is expensive to rebuild
per call — a fitted estimator, the feature order, the decision threshold. Feature
engineering stays functional: it is a stateless transformation, so a class there
would add ceremony without adding anything.

In [2]:
run("find src api tests scripts -name '*.py' | sort")

api/main.py
api/routes.py
api/schemas.py
scripts/__init__.py
scripts/logging_demo.py
src/config.py
src/data_quality.py
src/exceptions.py
src/feature_engineering.py
src/logging_config.py
src/predict.py
src/preprocessing.py
src/train.py
tests/__init__.py
tests/conftest.py
tests/test_api.py
tests/test_data_validation.py
tests/test_inference.py
tests/test_model_training.py



0

In [3]:
from src.train import ModelTrainer
from src.predict import Predictor

for cls in (ModelTrainer, Predictor):
    methods = [m for m in vars(cls) if not m.startswith("_")]
    print(f"{cls.__name__:14} {methods}")

ModelTrainer   ['fit', 'evaluate', 'overfit_small_batch', 'save']
Predictor      ['prepare', 'predict_single', 'predict_batch']


## 2. Research code vs production code

The same function, `map_features()`, before and after. The "before" is the
version submitted for Assignment I, preserved in `reports/before_refactor/`.

In [4]:
import re

def show(path, name):
    text = Path(path).read_text()
    m = re.search(rf"def {name}\(.*?(?=\n(?:# =|def |\Z))", text, re.S)
    print(m.group(0).rstrip() if m else "not found")

print("=" * 70)
print("BEFORE — research code (Assignment I)")
print("=" * 70)
show("reports/before_refactor/preprocessing_before.py", "map_features")

BEFORE — research code (Assignment I)
def map_features(df: pd.DataFrame):
    gender_map = {
        "Male": 1,
        "Female": 0
    }
    vehicle_age_map = {
        "< 1 Year": 0,
        "1-2 Year": 1,
        "> 2 Years": 2
    }
    damage_map = {
        "No": 0,
        "Yes": 1
    }
    df["Gender"] = df["Gender"].map(gender_map)
    df["Vehicle_Age"] = df["Vehicle_Age"].map(vehicle_age_map)
    df["Vehicle_Damage"] = df["Vehicle_Damage"].map(damage_map)
    return df


In [5]:
print("=" * 70)
print("AFTER — production code")
print("=" * 70)
show("src/preprocessing.py", "map_features")

AFTER — production code
def map_features(df: pd.DataFrame) -> pd.DataFrame:
    """Map the three categorical columns to integer codes.

    Previously this used a bare .map(), which turns any unrecognised value
    into NaN. XGBoost treats NaN as a legitimate missing value, so a record
    with Gender="Other" was scored and returned a confident probability
    instead of being rejected. We now validate first and fail loudly.
    """
    df = df.copy()
    validate_schema(df)

    df["Gender"] = df["Gender"].map(GENDER_MAP)
    df["Vehicle_Age"] = df["Vehicle_Age"].map(VEHICLE_AGE_MAP)
    df["Vehicle_Damage"] = df["Vehicle_Damage"].map(DAMAGE_MAP)
    return df


### The difference that matters

The research version has no unhappy path. `.map()` returns `NaN` for any value
not in the mapping, and XGBoost treats `NaN` as a legitimate missing value — so
a malformed record is scored rather than rejected. That is a reasonable trade in
a notebook where you can see the dataframe. It is not reasonable in a service,
where the caller cannot distinguish a good prediction from a prediction on
garbage.

In [6]:
import pandas as pd
from src.preprocessing import map_features
from src.exceptions import DataValidationError

record = pd.DataFrame([{
    "Gender": "Other", "Age": 45, "Driving_License": 1, "Region_Code": 28.0,
    "Previously_Insured": 0, "Vehicle_Age": "> 2 Years", "Vehicle_Damage": "Yes",
    "Annual_Premium": 35000.0, "Policy_Sales_Channel": 26.0, "Vintage": 150,
}])

# Research behaviour: silent NaN
print("Research version (.map only):")
print("  Gender ->", record["Gender"].map({"Male": 1, "Female": 0}).iloc[0], "(NaN — scored anyway)")

# Production behaviour: rejected
print("\nProduction version (validate then map):")
try:
    map_features(record)
except DataValidationError as exc:
    print("  DataValidationError:", exc)

Research version (.map only):
  Gender -> nan (NaN — scored anyway)

Production version (validate then map):
2026-08-12 23:14:50 | ERROR    | src.preprocessing:89 | Schema validation failed - column Gender contains unrecognised values ['Other'] (allowed: ['Male', 'Female'])


  DataValidationError: Column 'Gender' contains unrecognised values: ['Other']


## 3. Error handling and logging

`src/logging_config.py` configures the root logger once — INFO to console, DEBUG
to a rotating file at `logs/application.log`. Modules obtain loggers via
`get_logger(__name__)` so every record identifies its source.

Failures raise typed exceptions from `src/exceptions.py`, which the API maps to
status codes in one place. Implemented across five modules: `preprocessing`,
`train`, `predict`, `data_quality`, and the API layer.

In [7]:
run("python -m scripts.logging_demo")

2026-08-12 23:14:51 | INFO     | __main__:88 | === Logging demonstration started ===
2026-08-12 23:14:51 | INFO     | __main__:42 | --- INFO: normal operation ---
2026-08-12 23:14:52 | INFO     | src.preprocessing:61 | Loaded 381109 rows and 12 columns from dataset/train.csv
2026-08-12 23:14:52 | INFO     | src.preprocessing:100 | Schema validation passed for dataframe with shape (5000, 12)
2026-08-12 23:14:52 | INFO     | src.data_quality:46 | Missing-value check passed (worst column: 0.0000)
2026-08-12 23:14:52 | INFO     | __main__:51 | --- WARNING: recoverable anomalies ---
2026-08-12 23:14:52 | WARNING  | src.preprocessing:113 | Removed 50 duplicate rows (50.00% of input)
2026-08-12 23:14:52 | WARNING  | src.preprocessing:189 | 1 rows have an Age outside the range 18-100 and could not be binned
2026-08-12 23:14:52 | WARNING  | src.data_quality:103 | PSI=0.1797 indicates moderate drift (threshold 0.10) - monitor closely
2026-08-12 23:14:52 | INFO     | __main__:65 | --- ERROR: fail

0

In [8]:
print("Log levels emitted:")
run("python -m scripts.logging_demo 2>&1 | grep -oE 'INFO|WARNING|ERROR' | sort | uniq -c")
print("\nPersisted log file (last 5 lines):")
run("tail -5 logs/application.log")

Log levels emitted:


      8 ERROR
     10 INFO
      4 WARNING


Persisted log file (last 5 lines):
2026-08-12 23:14:53 | ERROR    | __main__:75 | Caught expected category failure: Column 'Gender' contains unrecognised values: ['Other']
2026-08-12 23:14:53 | ERROR    | src.data_quality:97 | PSI=12.5725 exceeds the significant-drift threshold (0.25) - retraining indicated
2026-08-12 23:14:53 | ERROR    | src.preprocessing:83 | Schema validation failed - missing columns: ['Gender', 'Driving_License', 'Region_Code', 'Previously_Insured', 'Vehicle_Age', 'Vehicle_Damage', 'Annual_Premium', 'Policy_Sales_Channel', 'Vintage']
2026-08-12 23:14:53 | ERROR    | __main__:84 | Caught expected validation failure: Missing required columns: ['Gender', 'Driving_License', 'Region_Code', 'Previously_Insured', 'Vehicle_Age', 'Vehicle_Damage', 'Annual_Premium', 'Policy_Sales_Channel', 'Vintage']
2026-08-12 23:14:53 | INFO     | __main__:92 | === Logging demonstration finished ===



0

## 4. Code formatting and linting

`black`, `isort` and `flake8` are configured in `pyproject.toml` and `.flake8`,
both committed so every group member gets identical results.

| | flake8 issues |
|---|---|
| Before | 107 |
| After | 0 |

In [9]:
print("BEFORE (captured prior to refactoring — reports/lint_before.txt):\n")
run("head -12 reports/lint_before.txt; echo '   ...'; grep 'Total issues' reports/lint_before.txt")

BEFORE (captured prior to refactoring — reports/lint_before.txt):

=== flake8 (BEFORE) ===
api/main.py:6:100: E501 line too long (141 > 99 characters)
api/main.py:10:27: W292 no newline at end of file
api/routes.py:13:1: E302 expected 2 blank lines, found 1
api/routes.py:15:100: E501 line too long (120 > 99 characters)
api/routes.py:15:110: E231 missing whitespace after ':'
api/routes.py:17:1: E302 expected 2 blank lines, found 1
api/routes.py:19:100: E501 line too long (119 > 99 characters)
api/routes.py:21:1: E302 expected 2 blank lines, found 1
api/routes.py:26:1: E302 expected 2 blank lines, found 1
api/routes.py:26:14: E201 whitespace after '('
api/routes.py:33:1: E302 expected 2 blank lines, found 1
   ...
Total issues: 107



0

In [10]:
print("AFTER:\n")
run("python -m flake8 src api tests streamlit scripts && echo 'flake8: no issues'")
run("python -m black --check src api tests streamlit scripts 2>&1 | tail -2")
run("python -m isort --check-only src api tests streamlit scripts && echo 'isort: imports correctly sorted'")

AFTER:



flake8: no issues



All done! ✨ 🍰 ✨
23 files would be left unchanged.

isort: imports correctly sorted



0

## 5. REST API design

| Method | Endpoint | Purpose | Codes |
|---|---|---|---|
| GET | `/` | Service banner | 200 |
| GET | `/health` | Liveness probe, reports `model_loaded` | 200 |
| GET | `/metrics` | Metrics recorded at training time | 200, 503 |
| POST | `/predict` | Score one customer | 200, 422, 503 |
| POST | `/batch-predict` | Score an uploaded CSV | 200, 422 |

Every route declares a `response_model`, so the OpenAPI schema is generated from
the code rather than maintained by hand. Exceptions map to status codes centrally
in `api/main.py`, keeping route bodies free of `try/except` noise.

In [11]:
from fastapi.testclient import TestClient
from api.main import app

client = TestClient(app)

valid = {
    "id": 1, "Gender": "Male", "Age": 45, "Driving_License": 1,
    "Region_Code": 28.0, "Previously_Insured": 0, "Vehicle_Age": "> 2 Years",
    "Vehicle_Damage": "Yes", "Annual_Premium": 35000.0,
    "Policy_Sales_Channel": 26.0, "Vintage": 150,
}

print("GET  /health  ->", client.get("/health").status_code, client.get("/health").json())
r = client.post("/predict", json=valid)
print("POST /predict ->", r.status_code, r.json())

2026-08-12 23:14:55 | INFO     | src.predict:44 | Predictor ready: 22 features, decision threshold 0.40


2026-08-12 23:14:55 | INFO     | httpx:1025 | HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


2026-08-12 23:14:55 | INFO     | httpx:1025 | HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


GET  /health  -> 200 {'status': 'ok', 'model_loaded': True, 'model_version': '1.0.0'}
2026-08-12 23:14:55 | INFO     | src.preprocessing:100 | Schema validation passed for dataframe with shape (1, 10)


2026-08-12 23:14:55 | INFO     | src.predict:90 | Scored record -> probability=0.7304 threshold=0.40 prediction=1


2026-08-12 23:14:55 | INFO     | httpx:1025 | HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


POST /predict -> 200 {'prediction': 1, 'probability': 0.7304, 'label': 'Interested', 'threshold': 0.4, 'model_version': '1.0.0'}


In [12]:
print("Input validation — each of these previously returned a confident prediction:\n")
for field, value in [("Gender", "Other"), ("Vehicle_Damage", "maybe"),
                     ("Age", 5), ("Annual_Premium", -100)]:
    resp = client.post("/predict", json={**valid, field: value})
    print(f"  {field}={value!r:10} -> HTTP {resp.status_code}")

Input validation — each of these previously returned a confident prediction:

2026-08-12 23:14:55 | INFO     | httpx:1025 | HTTP Request: POST http://testserver/predict "HTTP/1.1 422 Unprocessable Entity"


  Gender='Other'    -> HTTP 422
2026-08-12 23:14:55 | INFO     | httpx:1025 | HTTP Request: POST http://testserver/predict "HTTP/1.1 422 Unprocessable Entity"


  Vehicle_Damage='maybe'    -> HTTP 422
2026-08-12 23:14:55 | INFO     | httpx:1025 | HTTP Request: POST http://testserver/predict "HTTP/1.1 422 Unprocessable Entity"


  Age=5          -> HTTP 422
2026-08-12 23:14:55 | INFO     | httpx:1025 | HTTP Request: POST http://testserver/predict "HTTP/1.1 422 Unprocessable Entity"


  Annual_Premium=-100       -> HTTP 422


---
# Objective 2 — Quality Assurance

## 6 & 7. Test suite

Three test types are represented: **unit** (individual functions), **integration**
(full stack through `TestClient`), and **data validation** (schema and drift).

The ML-specific tests assert on *properties* rather than fixed numbers, so they
survive a retrain:

- **Overfit a small batch** — an unregularised model given 50 rows should nearly
  memorise them. Failure indicates a structural fault (features not reaching the
  model, labels misaligned) rather than a tuning problem.
- **Loss decreases** — a 200-estimator model must beat a 2-estimator one.
- **Directional** — prior vehicle damage must not lower predicted interest.
- **Invariance** — changing `id` must not change the prediction, which would
  indicate an identifier leaking into the feature matrix.

In [13]:
run("python -m pytest -v --no-header 2>&1 | grep -E 'PASSED|FAILED|passed|failed'")

tests/test_api.py::test_health_reports_model_loaded PASSED               [  2%]
tests/test_api.py::test_metrics_endpoint_returns_recorded_metrics PASSED [  4%]
tests/test_api.py::test_metrics_payload_satisfies_dashboard_contract PASSED [  6%]
tests/test_api.py::test_predict_returns_well_formed_response PASSED      [  9%]
tests/test_api.py::test_invalid_field_values_are_rejected[Gender-Other] PASSED [ 11%]
tests/test_api.py::test_invalid_field_values_are_rejected[Vehicle_Age-3 Years] PASSED [ 13%]
tests/test_api.py::test_invalid_field_values_are_rejected[Vehicle_Damage-maybe] PASSED [ 15%]
tests/test_api.py::test_invalid_field_values_are_rejected[Age-5] PASSED  [ 18%]
tests/test_api.py::test_invalid_field_values_are_rejected[Age-150] PASSED [ 20%]
tests/test_api.py::test_invalid_field_values_are_rejected[Annual_Premium--100] PASSED [ 22%]
tests/test_api.py::test_invalid_field_values_are_rejected[Previously_Insured-7] PASSED [ 25%]
tests/test_api.py::test_missing_required_field_is_reject

0

## 8a. Model quality metrics

Evaluated at the shared `DECISION_THRESHOLD` of 0.40, so the reported figures
describe the decision rule the API actually applies.

In [14]:
metrics = json.load(open("models/metrics.json"))
rows = [(k, metrics[k]) for k in
        ("roc_auc", "recall", "precision", "f1_score", "accuracy", "threshold")]
print(f"{'metric':<12} value")
print("-" * 24)
for k, v in rows:
    print(f"{k:<12} {v:.4f}")

print(f"\nTrain samples: {metrics['train_samples']:,}")
print(f"Test samples:  {metrics['test_samples']:,}")
print(f"\nConfusion matrix: {metrics['confusion_matrix']}")

metric       value
------------------------
roc_auc      0.8593
recall       0.8827
precision    0.3009
f1_score     0.4488
accuracy     0.7343
threshold    0.4000

Train samples: 342,998
Test samples:  38,111

Confusion matrix: [[23862, 9578], [548, 4123]]


Recall is deliberately prioritised over precision. A missed interested customer
is lost revenue; a wasted call is cheap. That trade-off is the reason the
threshold sits at 0.40 rather than the 0.50 default, and it maps directly to the
Assignment I goal of recall above 75%.

## 8b. Data quality metrics

Three gates: schema validation (hard pass/fail), missing-value rate (against a 5%
tolerance), and PSI drift against the training distribution.

In [15]:
from src.data_quality import missing_value_report, population_stability_index
from src.preprocessing import load_data, validate_schema

df = load_data("dataset/train.csv").head(20000)

print("1. Schema validation")
validate_schema(df, require_target=True)

print("\n2. Missing-value rate (worst 3 columns)")
report = missing_value_report(df)
for col, rate in sorted(report.items(), key=lambda x: -x[1])[:3]:
    print(f"   {col:<22} {rate:.4%}")

2026-08-12 23:14:59 | INFO     | src.preprocessing:61 | Loaded 381109 rows and 12 columns from dataset/train.csv


1. Schema validation
2026-08-12 23:14:59 | INFO     | src.preprocessing:100 | Schema validation passed for dataframe with shape (20000, 12)



2. Missing-value rate (worst 3 columns)
2026-08-12 23:14:59 | INFO     | src.data_quality:46 | Missing-value check passed (worst column: 0.0000)


   id                     0.0000%
   Gender                 0.0000%
   Age                    0.0000%


In [16]:
print("3. PSI drift — same distribution vs progressively shifted\n")
reference = df["Annual_Premium"]
for label, current in [
    ("identical",        reference),
    ("shifted +2,000",   reference + 2000),
    ("shifted +20,000",  reference + 20000),
    ("doubled +20,000",  reference * 2 + 20000),
]:
    psi = population_stability_index(reference, current)
    band = ("stable" if psi < 0.10 else
            "moderate drift" if psi < 0.25 else "significant drift")
    print(f"   {label:<18} PSI = {psi:7.4f}   {band}")

3. PSI drift — same distribution vs progressively shifted

2026-08-12 23:14:59 | INFO     | src.data_quality:109 | PSI=0.0000 - distribution stable


   identical          PSI =  0.0000   stable
2026-08-12 23:14:59 | INFO     | src.data_quality:109 | PSI=0.0350 - distribution stable


   shifted +2,000     PSI =  0.0350   stable
2026-08-12 23:14:59 | ERROR    | src.data_quality:97 | PSI=6.2576 exceeds the significant-drift threshold (0.25) - retraining indicated


   shifted +20,000    PSI =  6.2576   significant drift
2026-08-12 23:14:59 | ERROR    | src.data_quality:97 | PSI=9.4923 exceeds the significant-drift threshold (0.25) - retraining indicated


   doubled +20,000    PSI =  9.4923   significant drift


PSI is the metric that matters after deployment. The model does not degrade
because the code changed — it degrades because the customer population changed,
and nothing in the test suite can detect that.

## 9. Testing in production, and a security consideration

### Part A — Production experimentation

Offline metrics tell us how the model would have performed on historical
customers. They cannot tell us whether the marketing team converts more policies.
Three approaches close that gap, and they are complementary stages rather than
alternatives:

**Shadow deployment (week 1).** The challenger scores every live request but its
output is logged, not returned. Validates that it survives real traffic — real
latency, real payload shapes — at zero customer cost. Gate: no errors, p99
latency within the 500 ms NFR, PSI between score distributions below 0.25. This
is the stage that catches train/serve skew, the category the `Age == 18` crash
belonged to. It cannot validate whether predictions are any *good*, because no
customer is ever contacted on them.

**Canary release (week 2).** 5% of traffic to the challenger, using `/health` as
the readiness gate so a revision that cannot load its model never receives
traffic. Small blast radius, automatic rollback on elevated 5xx. Too small a
sample to settle anything statistically subtle.

**A/B test (one campaign cycle).** Random assignment, both arms acted on,
conversion compared. The only stage that answers the business question causally.

For this system specifically, the decision threshold is where most of the
campaign economics live — running 0.40 against 0.50 as two arms would measure
that business assumption directly.

### Part B — Security: input validation at the API boundary

**The vulnerability.** `api/schemas.py` originally typed the categorical fields
as plain `str`. A request carrying `Gender: "Other"` passed validation, reached
`.map()`, became `NaN`, and was scored by XGBoost as a missing value. The caller
received a confident-looking prediction on input the model had never been trained
to handle. Nothing was logged, because nothing had failed as far as the code was
concerned.

**Why this is a security issue.** It is a silent failure in a system that drives
spending decisions. A buggy upstream integration — or an attacker — can submit
out-of-distribution values and receive authoritative-looking scores. Because
`/batch-predict` accepted arbitrary CSVs, one malformed file could poison an
entire campaign's targeting list, and the only symptom would be a disappointing
conversion rate a month later. Silent wrong answers are worse than loud failures
because they are acted upon.

**Controls implemented.**

| Control | Where | Effect |
|---|---|---|
| `Literal` types on categoricals | `api/schemas.py` | Unknown categories rejected with 422 |
| Numeric bounds (`ge`/`le`) | `api/schemas.py` | Rejects `Age=5`, negative premiums |
| Schema re-validation in `map_features()` | `src/preprocessing.py` | Defence in depth for non-API callers |
| 5 MB upload / 10,000-row caps | `api/routes.py` | Bounds memory; DoS mitigation |
| File extension check | `api/routes.py` | Rejects non-CSV payloads |
| Typed exceptions → status codes | `api/main.py` | 422 client, 503 server; no internal detail leaked |

The layering is deliberate: pydantic guards the HTTP boundary, but Streamlit and
any future batch job call `Predictor` directly and would bypass it entirely.

**Next gap.** `/predict` is unauthenticated. For a model trained on customer data,
repeated querying is itself an information-disclosure risk — an attacker can probe
the decision boundary to infer the training population. An API key per consumer
plus rate limiting would address both that and unbounded usage cost.

### Demonstrated on the live deployment

Both services are running simultaneously — the Assignment I API and the
Assignment II API — so the fix can be shown against a real endpoint rather than
described.

```bash
# Assignment I API (before)
$ curl -X POST https://health-insurance-cross-sell-prediction.onrender.com/predict \
    -H "Content-Type: application/json" -d '{... "Gender": "Other" ...}'

{"prediction":1,"probability":0.7316,"label":"Interested"}

# Assignment II API (after)
$ curl -X POST https://seml2-group173.onrender.com/predict \
    -H "Content-Type: application/json" -d '{... "Gender": "Other" ...}'

{"detail":[{"type":"literal_error","loc":["body","Gender"],
"msg":"Input should be 'Male' or 'Female'","input":"Other"}]}
```

---
# Defects found and fixed

Four defects were found in the Assignment I codebase during this work. Each has
a regression test.

| # | Defect | Cause | Test |
|---|---|---|---|
| 1 | `Age = 18` crashed prediction | `pd.cut` excludes the left bin edge, so 18 became `NaN` and `OrdinalEncoder` raised. The API schema explicitly accepts `Age >= 18`. | `test_minimum_age_is_binned_not_dropped` |
| 2 | Invalid categories scored silently | `.map()` returns `NaN` for unknown keys; XGBoost treats `NaN` as missing | `test_unrecognised_category_is_rejected` |
| 3 | Reported metrics described an undeployed rule | `train.py` evaluated at 0.40; `predict.py` used `model.predict()`, hard-coded to 0.50 | `test_evaluate_uses_the_shared_decision_threshold` |
| 4 | `src/featureEngineering.py` was dead code | Divergent copy of `feature_engineering.py`; collided on case-insensitive filesystems | — (deleted) |

A fifth was introduced and caught *during* refactoring: removing
`train_samples`/`test_samples` from `metrics.json` broke the Streamlit dashboard
with a `KeyError`. All API tests passed, because they only tested the API —
`metrics.json` is a contract between two independently deployed components and
nothing covered that boundary. Test: `test_metrics_payload_satisfies_dashboard_contract`.

That fifth one is the most instructive. It is exactly the integration failure
mode that unit tests cannot catch, and it is why the suite includes end-to-end
tests through `TestClient` rather than only testing functions in isolation.

In [17]:
print("Defect 1 — Age = 18 no longer crashes")
print("  HTTP", client.post("/predict", json={**valid, "Age": 18}).status_code)

print("\nDefect 3 — training and inference share one threshold")
from src.config import DECISION_THRESHOLD
print(f"  config.DECISION_THRESHOLD = {DECISION_THRESHOLD}")
print(f"  metrics.json threshold    = {metrics['threshold']}")
print(f"  Predictor threshold       = {Predictor().threshold}")

print("\nDefect 5 — metrics.json satisfies the dashboard contract")
needed = {"model", "dataset", "train_samples", "test_samples", "accuracy",
          "precision", "recall", "f1_score", "roc_auc", "confusion_matrix"}
print("  all keys present:", needed <= set(metrics))

Defect 1 — Age = 18 no longer crashes
2026-08-12 23:14:59 | INFO     | src.preprocessing:100 | Schema validation passed for dataframe with shape (1, 10)


2026-08-12 23:14:59 | INFO     | src.predict:90 | Scored record -> probability=0.3901 threshold=0.40 prediction=0


2026-08-12 23:14:59 | INFO     | httpx:1025 | HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


  HTTP 200

Defect 3 — training and inference share one threshold
  config.DECISION_THRESHOLD = 0.4
  metrics.json threshold    = 0.4
2026-08-12 23:14:59 | INFO     | src.predict:44 | Predictor ready: 22 features, decision threshold 0.40


  Predictor threshold       = 0.4

Defect 5 — metrics.json satisfies the dashboard contract
  all keys present: True


---
## Summary

| Objective | Requirement | Evidence |
|---|---|---|
| 1.1 | Modular OOP/functional design | `src/` module tree, `ModelTrainer`, `Predictor` |
| 1.2 | Research vs production | `Notebooks/model.ipynb` vs `src/preprocessing.py` |
| 1.3 | Logging & error handling | 5 modules, 3 levels, typed exceptions |
| 1.4 | Formatting & linting | 107 → 0 flake8 issues |
| 1.5 | REST API | 5 endpoints, schemas, status codes |
| 2.6 | 2+ test types | unit, integration, data validation |
| 2.7a | Training tests | overfit small batch, loss decreases |
| 2.7b | Inference tests | shape/range, directional, invariance |
| 2.8a | Model quality | ROC-AUC 0.8593, recall 0.8827, F1, precision, accuracy |
| 2.8b | Data quality | schema validation, missing values, PSI drift |
| 2.9 | Production testing + security | shadow → canary → A/B; input validation |

**44 tests passing · flake8 clean · ROC-AUC 0.8593**